In [1]:
import pandas as pd
import geopandas as gp
import time
from shapely.geometry import LineString

In [6]:
# Loading the data into a dataframe
path = r'C:\Users\andre\OneDrive\Dokumente\UNI\Master_Digital_Humanities\S3_WS2024\LP_Data_Analysis_Project\data'
stdb = pd.read_spss(path+'\\tastdb-exp-2020.sav',
                    usecols=[
    #Voyage-Info
    "VOYAGEID", #ID for identification
    "SHIPNAME", #Name of vessel
    "DATEDEP", #Date that voyage began
    "DATEEND", #Date when voyage completed
    "YEAR5", #5-year period in which voyage occurred
    "YEAR10", #Decade in which voyage occurred
    "YEAR25", #Quarter-century in which voyage occurred
    "YEAR100", #Century in which voyage occurred
    "NATINIMP", #Imputed country in which ship registered
    "YRCONS", #Year of vessel’s construction
    "YRREG", #Year of vessel’s registration
    "YEARDEP", #Year voyage began (imputed)
    "CAPTAINA", #First captain’s name
    "CAPTAINB", #Second captain’s name
    "CAPTAINC", #Third captain’s name
    "JAMCASPR", #Average price of slaves standardized on sterling cash price of prime slaves sold in Jamaica
    "TONMOD", #Tonnage standardized on British measured tons, 1773-1835
    "PLACCONS", #Place where vessel constructed
    "PLACREG", #Place where vessel registered
    "PORTRET", #Port at which voyage ended
    "PTDEPIMP", #Imputed port where voyage began
    "FATE3", #Outcome of voyage if vessel captured, where there some by the British???
    "SLADVOY", #Slaves deaths between African and the Americas, for Animation??
    "OWNERA", #First owner of venture
    "OWNERB", #
    "OWNERC", #
    "OWNERD", #
    "OWNERE", #
    "OWNERF", #
    "OWNERG", #
    "OWNERH", #
    "OWNERI", #
    "OWNERJ", #
    "OWNERK", #
    "OWNERL", #
    "OWNERM", #
    "OWNERN", #
    "OWNERO", #
    "OWNERP", #Sixteenth owner of venture

    #Embarkation
    "YEARAF", #Year departed Africa (imputed)
    "MJBYPTIMP", #Imputed principal place of slave purchase
    "D1SLATRC", #Year that slave purchase began
    "DATEBUY", #Date that slave purchase began
    "SLAXIMP", #Imputed total slaves embarked
    "ADLT1IMP", #Derived number of adult embarked -> insgesamt, daher unnötig??

    #Disembarkation
    "YEARAM", #Year of arrival at port of disembarkation (imputed)
    "MJSLPTIMP", #Imputed principal place of slave disembarkation, DAP for Animation
    "MJSELIMP", #Imputed principal region of slave disembarkation
    "MJSELIMP1", #Imputed broad region of slave disembarkation
    "SLA1PORT", #First place of slave landing
    "ADPSALE1", #Second place of slave landing GIS, DAP
    "ADPSALE2", #Third place of slave landing
    "DATELAND1", #Date that slaves landed at first place (YYYY-MM-DD)
    "DATELAND2", #Date that slaves landed at second place
    "DATELAND3", #Date that slaves landed at third place
    "SLAMIMP", #Imputed total slaves disembarked
    "SLAARRIV", #Total slaves arrived at first port of disembarkation
    "SLAS32", #Slaves disembarked at first place
    "SLAS36", #Slaves disembarked at second place
    "SLAS39", #Slaves disembarked at third place
    "MALE3", #Males disembarked at first place of landing
    "MALE6", #Males disembarked at second place of landing
    "FEMALE3", #Females disembarked at first place of landing
    "FEMALE6", #Females disembarked at second place of landing
    "MEN3", #Men disembarked at first place of landing
    "MEN6", #Men disembarked at second place of landing
    "WOMEN3", #Women disembarked at first place of landing
    "WOMEN6", #Women disembarked at second place of landing
    "ADLT3IMP", #Derived number of adults landed
    "ADULT3", #Adults disembarked at first place of landing, GIS
    "ADULT6", #Adults disembarked at second place of landing, GIS
    "ADULT7", #Adults disembarked at third place of landing, GIS
    "INFANT3", #Infants disembarked at first place of landing
    "INFANT6", #Infants disembarked at second place of landing
    "BOY3", #Boys disembarked at first place of landing
    "BOY6", #Boys disembarked at second place of landing
    "GIRL3", #Girls disembarked at first place of landing
    "GIRL6", #Girls disembarked at second place of landing
    "CHILD3", #Children disembarked at first place of landing
    "CHILD6", #Children disembarked at second place of landing

    #For Animation
    "TSLAVESD", #Total slaves on board at departure from last slaving port
    "TSLMTIMP", #Derived number of slaves embarked for mortality calculation
    "VOY1IMP", #Voyage length from home port to disembarkation (days)
    "VOY2IMP", #Voyage length from leaving Africa to disembarkation (days)
    "VOYAGE", #Length of Middle Passage in days
    "VYMRTIMP", #Derived slave deaths during Middle Passage
    "VYMRTRAT" #Slave mortality rate (slave deaths / slaves embarked)

             ]
                    
                    
                    )

In [7]:
#Change datatype of ID-column and set it as index
stdb = stdb.astype({'VOYAGEID': int})
stdb = stdb.set_index("VOYAGEID")

In [8]:
#extract categorical columns
categorical_columns = stdb.select_dtypes(include=['category']).columns

#convert categorical columns to str (geopackage can't deal with that)
for column in categorical_columns:
    stdb[column] = stdb[column].astype(str)

In [9]:
#Set date-columns as datetime
stdb[["DATEDEP","DATEEND","DATEBUY","DATELAND1","DATELAND2","DATELAND3"]] = stdb[["DATEDEP","DATEEND","DATEBUY","DATELAND1","DATELAND2","DATELAND3"]].apply(lambda x: pd.to_datetime(x,errors = 'coerce', format = '%Y-%m-%d'))

In [10]:
#delete string 'year ' from year columns
for column in ["YEAR5","YEAR10","YEAR25"]:
   stdb[column] =  stdb[column].str.replace("years ",'')

In [11]:
stdb.describe()

,ADLT1IMP,ADLT3IMP,ADULT3,ADULT6,ADULT7,BOY3,BOY6,CHILD3,CHILD6,D1SLATRC,...,VYMRTIMP,VYMRTRAT,WOMEN3,WOMEN6,YEAR100,YEARAF,YEARAM,YEARDEP,YRCONS,YRREG
count,1554.000000,2865.000000,55.000000,0.0,4206.000000,2936.000000,26.000000,256.000000,2.000000,8834.000000,...,6517.000000,6477.000000,3029.000000,29.000000,36108.000000,36108.000000,36108.000000,36108.000000,6261.000000,4917.000000
mean,297.872587,184.267714,242.745455,NaN,220.416072,38.429837,30.461538,18.902344,56.000000,1763.743378,...,43.077183,0.121785,63.000990,53.448276,1715.774898,1764.264983,1764.327185,1763.864296,1767.698610,1747.184259
min,5.000000,0.000000,16.000000,NaN,0.000000,1.000000,1.000000,1.000000,43.000000,1526.000000,...,0.000000,0.000000,1.000000,4.000000,1500.000000,1514.000000,1514.000000,1514.000000,1548.000000,1526.000000
25%,173.000000,97.000000,158.500000,NaN,112.000000,14.000000,15.500000,2.000000,49.500000,1739.000000,...,7.000000,0.030000,29.000000,23.000000,1700.000000,1732.000000,1732.000000,1732.000000,1750.000000,1745.000000
50%,293.000000,167.000000,235.000000,NaN,190.500000,29.000000,29.000000,8.000000,56.000000,1768.000000,...,21.000000,0.070000,55.000000,57.000000,1700.000000,1773.000000,1773.000000,1772.000000,1769.000000,1763.000000
75%,402.750000,251.000000,323.000000,NaN,307.000000,52.000000,39.000000,26.000000,62.500000,1792.000000,...,52.000000,0.160000,88.000000,78.000000,1800.000000,1806.000000,1806.000000,1806.000000,1787.000000,1777.000000
max,881.000000,806.000000,513.000000,NaN,881.000000,367.000000,81.000000,199.000000,69.000000,1864.000000,...,998.000000,1.000000,268.000000,147.000000,1800.000000,1866.000000,1866.000000,1866.000000,1858.000000,1860.000000
std,157.377233,113.339357,115.242613,NaN,141.548093,34.652136,21.722303,28.190011,18.384776,40.079536,...,64.767171,0.148845,44.139733,35.735723,68.603335,59.487223,59.468134,59.519535,28.117804,56.800496


In [12]:
# Convert float columns to integers
columns_to_convert = [
    "YEARDEP",
    "YRCONS",
    "YRREG",
    "SLADVOY",
    "YEARAF",
    "D1SLATRC",
    "SLAXIMP",
    "ADLT1IMP",
    "YEARAM",
    "SLAS32",
    "SLAS36",
    "SLAS39",
    "MALE3",
    "MALE6",
    "FEMALE3",
    "FEMALE6",
    "MEN3",
    "MEN6",
    "WOMEN3",
    "WOMEN6",
    "ADLT3IMP",
    "ADULT3",
    "ADULT6",
    "ADULT7",
    "INFANT3",
    "INFANT6",
    "BOY3",
    "BOY6",
    "GIRL3",
    "GIRL6",
    "CHILD3",
    "CHILD6",
    "TSLAVESD",
    "TSLMTIMP",
    "VOY1IMP",
    "VOY2IMP",
    "VOYAGE",
    "VYMRTIMP",
    "YEAR100"
    ]

for column in columns_to_convert:
    if stdb[column].isna().sum() == 0:
        stdb[column] = stdb[column].astype('int')


In [14]:
# import gazetteer for the toponym resolution
ports = pd.read_csv(path+"\\slave-voyages-ports.csv")

In [15]:
#place-columns (needed for toponym resultion)
place_columns=[
    "PLACCONS", #Place where vessel constructed
    "PLACREG", #Place where vessel registered
    "PTDEPIMP", #Imputed port where voyage began
    "PORTRET", #Port at which voyage ended

    "MJBYPTIMP", #Imputed principal place of slave purchase

    "MJSLPTIMP", #Imputed principal place of slave disembarkation
    "MJSELIMP", #Imputed principal region of slave disembarkation
    "MJSELIMP1", #Imputed broad region of slave disembarkation
    "SLA1PORT", #First place of slave landing
    "ADPSALE1", #Second place of slave landing
    "ADPSALE2" #Third place of slave landing
    ]

In [12]:
#toponym resolution for columns in place_columns with ports-dataset (ca. 130s)
start_time = time.time()
for column in place_columns:
    column_name = column
    #longitude
    stdb[column_name+'_lng'] = stdb.apply(lambda x: ports.loc[ports['port'] == x[column], 'long'].reset_index(drop=True).values[0] if x[column] in ports['port'].values else None, axis=1)
    #latitude
    stdb[column_name+'_lat'] = stdb.apply(lambda x: ports.loc[ports['port'] == x[column], 'lat'].reset_index(drop=True).values[0] if x[column] in ports['port'].values else None, axis=1)
    print(column+ " finished: "+str(round(time.time() - start_time,2))+"s")

PLACCONS finished: 8.66s
PLACREG finished: 16.35s
PTDEPIMP finished: 40.4s
PORTRET finished: 50.4s
MJBYPTIMP finished: 73.57s
MJSLPTIMP finished: 96.43s
MJSELIMP finished: 98.86s
MJSELIMP1 finished: 101.3s
SLA1PORT finished: 121.0s
ADPSALE1 finished: 124.83s
ADPSALE2 finished: 127.71s


In [13]:
#Calculate Ratio of slaves per port
# Define the columns for each demographic group at Port 1 and Port 2
port1_columns = ['MEN3', 'WOMEN3', 'BOY3', 'GIRL3']
port2_columns = ['MEN6', 'WOMEN6', 'BOY6', 'GIRL6']

# Ensure there are no missing values (replace NaN with 0)
stdb[port1_columns + port2_columns] = stdb[port1_columns + port2_columns].fillna(0)

# Calculate the total slave count at Port 1 and Port 2
stdb['TOTAL_PORT1'] = stdb[port1_columns].sum(axis=1)
stdb['TOTAL_PORT2'] = stdb[port2_columns].sum(axis=1)

# Calculate ratios for Port 1
for col in port1_columns:
    stdb[f"{col}_RATIO_PORT1"] = stdb[col] / stdb['TOTAL_PORT1']

# Calculate ratios for Port 2
for col in port2_columns:
    stdb[f"{col}_RATIO_PORT2"] = stdb[col] / stdb['TOTAL_PORT2']

# Replace NaN ratios (caused by division by 0) with 0
ratio_columns = [f"{col}_RATIO_PORT1" for col in port1_columns] + [f"{col}_RATIO_PORT2" for col in port2_columns]
stdb[ratio_columns] = stdb[ratio_columns].fillna(0)


In [14]:
stdb

,ADLT1IMP,ADLT3IMP,ADPSALE1,ADPSALE2,ADULT3,ADULT6,ADULT7,BOY3,BOY6,CAPTAINA,...,TOTAL_PORT1,TOTAL_PORT2,MEN3_RATIO_PORT1,WOMEN3_RATIO_PORT1,BOY3_RATIO_PORT1,GIRL3_RATIO_PORT1,MEN6_RATIO_PORT2,WOMEN6_RATIO_PORT2,BOY6_RATIO_PORT2,GIRL6_RATIO_PORT2
VOYAGEID,,,,,,,,,,,,,,,,,,,,,
1,NaN,NaN,nan,nan,NaN,NaN,NaN,0.0,0.0,"Dias, Manoel José",...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,NaN,NaN,nan,nan,NaN,NaN,NaN,0.0,0.0,"Mata, José Maria da",...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,NaN,NaN,nan,nan,NaN,NaN,NaN,0.0,0.0,"Ferreira, José dos Santos",...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,NaN,NaN,nan,nan,NaN,NaN,NaN,0.0,0.0,"Dias, Domingos Francisco",...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
5,NaN,NaN,nan,nan,NaN,NaN,NaN,0.0,0.0,,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
900233,NaN,NaN,nan,nan,NaN,NaN,NaN,0.0,0.0,,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
900234,NaN,NaN,nan,nan,NaN,NaN,NaN,0.0,0.0,,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
900235,NaN,NaN,nan,nan,NaN,NaN,NaN,0.0,0.0,,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [15]:
#adult/child-ratio
stdb['ADULTS_PORT1'] = stdb['MEN3_RATIO_PORT1'] + stdb['WOMEN3_RATIO_PORT1']
stdb['CHILDREN_PORT1'] = stdb['BOY3_RATIO_PORT1'] + stdb['GIRL3_RATIO_PORT1']
stdb['ADULTS_PORT2'] = stdb['MEN6_RATIO_PORT2'] + stdb['WOMEN6_RATIO_PORT2']
stdb['CHILDREN_PORT2'] = stdb['BOY6_RATIO_PORT2'] + stdb['GIRL6_RATIO_PORT2']


In [19]:
#Export whole df to csv
stdb.to_csv('data\\stdb.csv', index=False, header=True)

In [16]:
#columns which every geopackage-file will need
columns_for_all = [
    #Voyage-Info
    #"VOYAGEID", #ID for identification
    "SHIPNAME", #Name of vessel
    "YEAR5", #5-year period in which voyage occurred
    "YEAR10", #Decade in which voyage occurred
    "YEAR25", #Quarter-century in which voyage occurred
    "YEAR100", #Century in which voyage occurred
    "NATINIMP", #Imputed country in which ship registered
    "YEARDEP", #Year voyage began (imputed)
    "YEARAM", # year of slave landing
    "CAPTAINA", #First captain’s name
    "CAPTAINB", #Second captain’s name
    "CAPTAINC", #Third captain’s name
    "JAMCASPR", #Average price of slaves standardized on sterling cash price of prime slaves sold in Jamaica
    "TONMOD", #Tonnage standardized on British measured tons, 1773-1835
    "FATE3", #Outcome of voyage if vessel captured, where there some by the British???
    "TSLAVESD" #Total slaves on board at departure from last slaving port
]

In [17]:
# columns with geodata
columns_geo = [
    "PLACCONS", #Place where vessel constructed
    "PLACREG", #Place where vessel registered
    "PTDEPIMP", #Imputed port where voyage began
    "PORTRET", #Port at which voyage ended

    #"MJBYPTIMP", #Imputed principal place of slave purchase

    #"MJSLPTIMP", #Imputed principal place of slave disembarkation
    #"MJSELIMP", #Imputed principal region of slave disembarkation
    #"MJSELIMP1", #Imputed broad region of slave disembarkation
    "SLA1PORT", #First place of slave landing
    "ADPSALE1", #Second place of slave landing
    "ADPSALE2" #Third place of slave landing
]

In [18]:
#specific columns for export per geo-column
columns_specific = [
    [
        "DATEDEP", #Date that voyage began
        "DATEEND", #Date when voyage completed
        "YRCONS", #Year of vessel’s construction
        "OWNERA", #First owner of venture
        "OWNERB", #
        "OWNERC", #
        "OWNERD", #
        "OWNERE", #
        "OWNERF", #
        "OWNERG", #
        "OWNERH", #
        "OWNERI", #
        "OWNERJ", #
        "OWNERK", #
        "OWNERL", #
        "OWNERM", #
        "OWNERN", #
        "OWNERO", #
        "OWNERP", #Sixteenth owner of venture
    ], #PLACCONS #Place where vessel constructed
    [
        "PTDEPIMP", #Imputed port where voyage began
        "DATEDEP", #Date that voyage began
        "DATEEND", #Date when voyage completed
        "YRREG", #Year of vessel’s construction
	    "PLACCONS", #Place where vessel constructed
        "OWNERA", #First owner of venture
        "OWNERB", #
        "OWNERC", #
        "OWNERD", #
        "OWNERE", #
        "OWNERF", #
        "OWNERG", #
        "OWNERH", #
        "OWNERI", #
        "OWNERJ", #
        "OWNERK", #
        "OWNERL", #
        "OWNERM", #
        "OWNERN", #
        "OWNERO", #
        "OWNERP", #Sixteenth owner of venture
    ], #PLACREG #Place where vessel registered
    [
        "DATEDEP", #Date that voyage began 
        "YRCONS", #Year of vessel’s construction
        "YEARAF", #Year departed Africa (imputed)
        "MJBYPTIMP", #Imputed principal place of slave purchase
        "D1SLATRC", #Year that slave purchase began
        "DATEBUY", #Date that slave purchase began
        "SLAXIMP", #Imputed total slaves embarked
        "ADLT1IMP", #Derived number of adult embarked -> insgesamt, daher unnötig??
    ], #PTDEPIMP #Imputed port where voyage began
    [
        "DATEEND", #Date when voyage completed 
            #Disembarkation
        "MJSLPTIMP", #Imputed principal place of slave disembarkation, DAP for Animation
        "MJSELIMP", #Imputed principal region of slave disembarkation
        "MJSELIMP1", #Imputed broad region of slave disembarkation
        "SLA1PORT", #First place of slave landing
        "ADPSALE1", #Second place of slave landing GIS, DAP
        "ADPSALE2", #Third place of slave landing
        "DATELAND1", #Date that slaves landed at first place (YYYY-MM-DD)
        "DATELAND2", #Date that slaves landed at second place
        "DATELAND3", #Date that slaves landed at third place
        "SLAMIMP", #Imputed total slaves disembarked
        "SLAARRIV", #Total slaves arrived at first port of disembarkation
        "SLAS32", #Slaves disembarked at first place
        "SLAS36", #Slaves disembarked at second place
        "SLAS39", #Slaves disembarked at third place
        "MALE3", #Males disembarked at first place of landing
        "MALE6", #Males disembarked at second place of landing
        "FEMALE3", #Females disembarked at first place of landing
        "FEMALE6", #Females disembarked at second place of landing
        "MEN3", #Men disembarked at first place of landing
        "MEN6", #Men disembarked at second place of landing
        "WOMEN3", #Women disembarked at first place of landing
        "WOMEN6", #Women disembarked at second place of landing
        "ADLT3IMP", #Derived number of adults landed
        "ADULT3", #Adults disembarked at first place of landing, GIS
        "ADULT6", #Adults disembarked at second place of landing, GIS
        "ADULT7", #Adults disembarked at third place of landing, GIS
        "INFANT3", #Infants disembarked at first place of landing
        "INFANT6", #Infants disembarked at second place of landing
        "BOY3", #Boys disembarked at first place of landing
        "BOY6", #Boys disembarked at second place of landing
        "GIRL3", #Girls disembarked at first place of landing
        "GIRL6", #Girls disembarked at second place of landing
        "CHILD3", #Children disembarked at first place of landing
        "CHILD6", #Children disembarked at second place of landing
    ], #PORTRET #Port at which voyage ended
    [
                    #Disembarkation
        "MJSLPTIMP", #Imputed principal place of slave disembarkation, DAP for Animation
        "MJSELIMP", #Imputed principal region of slave disembarkation
        "MJSELIMP1", #Imputed broad region of slave disembarkation
        "DATELAND1", #Date that slaves landed at first place (YYYY-MM-DD)
        "SLAARRIV", #Total slaves arrived at first port of disembarkation
        "SLAS32", #Slaves disembarked at first place
        "MALE3", #Males disembarked at first place of landing
        "FEMALE3", #Females disembarked at first place of landing
        "MEN3", #Men disembarked at first place of landing
        "WOMEN3", #Women disembarked at first place of landing
        "ADULT3", #Adults disembarked at first place of landing, GIS
        "INFANT3", #Infants disembarked at first place of landing
        "BOY3", #Boys disembarked at first place of landing
        "GIRL3", #Girls disembarked at first place of landing
        "CHILD3", #Children disembarked at first place of landing
        "ADULTS_PORT1",
        "MEN3_RATIO_PORT1",
        "WOMEN3_RATIO_PORT1",
        "CHILDREN_PORT1",
        "BOY3_RATIO_PORT1",
        "GIRL3_RATIO_PORT1"
    ], #SLA1PORT #First place of slave landing
    [
                    #Disembarkation
        "DATELAND2", #Date that slaves landed at second place
        "SLAS36", #Slaves disembarked at second place
        "MALE6", #Males disembarked at second place of landing
        "FEMALE6", #Females disembarked at second place of landing
        "MEN6", #Men disembarked at second place of landing
        "WOMEN6", #Women disembarked at second place of landing
        "ADULT6", #Adults disembarked at second place of landing, GIS
        "INFANT6", #Infants disembarked at second place of landing
        "BOY6", #Boys disembarked at second place of landing
        "GIRL6", #Girls disembarked at second place of landing
        "CHILD6", #Children disembarked at second place of landing
        "ADULTS_PORT2",
        "MEN6_RATIO_PORT2",
        "WOMEN6_RATIO_PORT2",
        "CHILDREN_PORT2",
        "BOY6_RATIO_PORT2",
        "GIRL6_RATIO_PORT2",
    ], #ADPSALE1 #Second place of slave landing
    [
                    #Disembarkation
        "DATELAND3", #Date that slaves landed at third place
        "SLAS39", #Slaves disembarked at third place
        "ADULT7", #Adults disembarked at third place of landing, GIS
    ] #ADPSALE2 #Third place of slave landing

]

In [20]:
#loop through columns_geo, filtered for relevant columns, export as single geopackage-files 
start_time = time.time()
for i in range(0,len(columns_geo)):
    #relevant columns
    geocolumn = columns_geo[i]
    relevant_columns = columns_for_all + columns_specific[i]
    relevant_columns.insert(0,columns_geo[i])
    relevant_columns.append(geocolumn+"_lng")
    relevant_columns.append(geocolumn+"_lat")

    #filter dataframe with relevant columns
    stdb_filter_columns = stdb[relevant_columns]

    # filter data where geometry is not Null
    stdb_filtered_geo = stdb_filter_columns[(stdb_filter_columns[[geocolumn+'_lng', geocolumn+'_lat']] != "0").all(axis=1) & stdb_filter_columns[[geocolumn+'_lng', geocolumn+'_lat']].notna().all(axis=1)]

    #dataframe to geodataframe
    gdf = gp.GeoDataFrame(stdb_filtered_geo, geometry=gp.points_from_xy(stdb_filtered_geo[geocolumn+'_lng'], stdb_filtered_geo[geocolumn+'_lat']), crs="EPSG:4326")
    
    #export gdf as geopackage
    gdf.to_file(f"data\\{geocolumn}.gpkg", driver="GPKG")

    print(geocolumn+ " finished: "+str(round(time.time() - start_time,2))+"s")

PLACCONS finished: 0.64s
PLACREG finished: 1.23s
PTDEPIMP finished: 2.8s
PORTRET finished: 3.74s
SLA1PORT finished: 5.23s
ADPSALE1 finished: 5.51s
ADPSALE2 finished: 5.74s


In [25]:
#filter data for SEL->DISEMB
columns_specific = [
    "DATEDEP", #Date that voyage began
    "DATEEND", #Date when voyage completed
    "SLADVOY", #Slaves deaths between African and the Americas, for Animation??
    "YEARAF", #Year departed Africa (imputed)
    "MJBYPTIMP", #Imputed principal place of slave purchase
    "DATEBUY", #Date that slave purchase began
    "MJSLPTIMP", #Imputed principal place of slave disembarkation, DAP for Animation
    "SLAMIMP", #Imputed total slaves disembarked
    "TSLMTIMP", #Derived number of slaves embarked for mortality calculation
    "VOY1IMP", #Voyage length from home port to disembarkation (days)
    "VOY2IMP", #Voyage length from leaving Africa to disembarkation (days)
    "VOYAGE", #Length of Middle Passage in days
    "VYMRTIMP", #Derived slave deaths during Middle Passage
    "VYMRTRAT", #Slave mortality rate (slave deaths / slaves embarked)
    "MJBYPTIMP_lng",
    "MJBYPTIMP_lat",
    "MJSLPTIMP_lng",
    "MJSLPTIMP_lat"
]

#filter df for relevant columns
relevant_columns = columns_for_all + columns_specific
stdb_filtered = stdb[relevant_columns]

#Export df to csv
stdb_filtered.to_csv('data\\stdb_BYSEL.csv', index=False, header=True)

# Apply additional filter: Remove rows where lat or lng are null
stdb_filtered2 = stdb_filtered.dropna(subset=["MJBYPTIMP_lng", "MJBYPTIMP_lat", "MJSLPTIMP_lng", "MJSLPTIMP_lat"])

# Export filtered DataFrame to CSV
stdb_filtered2.to_csv('data\\stdb_BYSEL_FILTER.csv', index=False, header=True)


In [28]:
# Create LineString geometries from start and end coordinates
geometry = stdb_filtered2.apply(
    lambda row: LineString([
        (row["MJBYPTIMP_lng"], row["MJBYPTIMP_lat"]),  # Start point
        (row["MJSLPTIMP_lng"], row["MJSLPTIMP_lat"])   # End point
    ]), axis=1
)

# Convert DataFrame to GeoDataFrame
gdf = gp.GeoDataFrame(stdb_filtered2, geometry=geometry)

# Set CRS (Coordinate Reference System) to WGS 84 (EPSG:4326)
gdf.set_crs(epsg=4326, inplace=True)

# Export to GeoPackage
output_path = "data\\stdb_lines.gpkg"
gdf.to_file(output_path, layer="lines", driver="GPKG")

print(f"GeoPackage saved at {output_path}")


GeoPackage saved at data\stdb_lines.gpkg


In [ ]:
#file for ports-SEL
stdb_SEL= stdb[["MJBYPTIMP","MJBYPTIMP_lng","MJBYPTIMP_lat"]]

#filter data where geometry is not Null
stdb_filtered_geo = stdb_SEL[(stdb_SEL[['MJBYPTIMP_lng', 'MJBYPTIMP_lat']] != "0").all(axis=1) & stdb_SEL[['MJBYPTIMP_lng', 'MJBYPTIMP_lat']].notna().all(axis=1)]

#drop duplicate rows
stdb_filtered_geo = stdb_filtered_geo.drop_duplicates()

#dataframe to geodataframe
gdf = gp.GeoDataFrame(stdb_filtered_geo, geometry=gp.points_from_xy(stdb_filtered_geo['MJBYPTIMP_lng'], stdb_filtered_geo['MJBYPTIMP_lat']), crs="EPSG:4326")

#export gdf as geopackage
gdf.to_file("data\\stdb_SEL.gpkg", driver="GPKG")

In [ ]:
#file for ports-BUY
stdb_BUY= stdb[["MJSLPTIMP", "MJSLPTIMP_lng", "MJSLPTIMP_lat"]]

#filter data where geometry is not Null
stdb_filtered_geo = stdb_BUY[(stdb_BUY[['MJSLPTIMP_lng', 'MJSLPTIMP_lat']] != "0").all(axis=1) & stdb_BUY[['MJSLPTIMP_lng', 'MJSLPTIMP_lat']].notna().all(axis=1)]

#drop duplicate rows
stdb_filtered_geo = stdb_filtered_geo.drop_duplicates()

#dataframe to geodataframe
gdf = gp.GeoDataFrame(stdb_filtered_geo, geometry=gp.points_from_xy(stdb_filtered_geo['MJSLPTIMP_lng'], stdb_filtered_geo['MJSLPTIMP_lat']), crs="EPSG:4326")

#export gdf as geopackage
gdf.to_file("data\\stdb_BUY.gpkg", driver="GPKG")

In [ ]:
# List of file paths for datasets
datasets_name = ["ADPSALE1", "ADPSALE2", "SLA1PORT"]

In [ ]:
# Loop through each dataset
for dataset in datasets_name:
    # 1. Read the dataset
    ports_sel = gp.read_file(f"data\\{dataset}.gpkg")
    
    # 2. One-hot encode the 'NATINIMP' column
    country_counts = pd.get_dummies(ports_sel[[dataset, 'NATINIMP', dataset+'_lng', dataset+'_lat']], columns=['NATINIMP'])
    
    # 3. Group by 'ADPSALE2' and aggregate the columns
    country_sums = country_counts.groupby(dataset).agg({
        **{col: 'sum' for col in country_counts.columns if col.startswith('NATINIMP_')},  # Sum for one-hot encoded columns
        dataset+'_lng': 'first',  # Keep the first entry of the longitude column
        dataset+'_lat': 'first',  # Keep the first entry of the latitude column
    }).reset_index()
    
    # 4. Add a column to count all countries together
    country_columns = [col for col in country_sums.columns if col.startswith('NATINIMP_')]
    country_sums['countries_SUM'] = country_sums[country_columns].sum(axis=1)
    
    # 5. Rename columns by removing the "NATINIMP_" prefix
    country_sums = country_sums.rename(
        columns=lambda col: col.replace("NATINIMP_", "") if col.startswith("NATINIMP_") else col
    )
    
    # 6. Convert DataFrame to GeoDataFrame
    gdf = gp.GeoDataFrame(
        country_sums,
        geometry=gp.points_from_xy(country_sums[dataset+'_lng'], country_sums[dataset+'_lat']),
        crs="EPSG:4326"
    )
    
    # 7. Export the GeoDataFrame to a GeoPackage
    output_path = f"data\\{dataset}_ports_SUM.gpkg"
    gdf.to_file(output_path, driver="GPKG")
    print(f"Processed and saved: {output_path}")


In [10]:
ports_sel.head(3)

,VOYAGEID,SLA1PORT,SHIPNAME,YEAR5,YEAR10,YEAR25,YEAR100,NATINIMP,YEARDEP,YEARAM,...,CHILD3,ADULTS_PORT1,MEN3_RATIO_PORT1,WOMEN3_RATIO_PORT1,CHILDREN_PORT1,BOY3_RATIO_PORT1,GIRL3_RATIO_PORT1,SLA1PORT_lng,SLA1PORT_lat,geometry
0,1,"Bahia, port unspecified",Pastora de Lima,1816-20,1811-20,1801-25,1800,Portugal / Brazil,1816,1817,...,NaN,0.0,0.0,0.0,0.0,0.0,0.0,-38.51083,-12.97111,POINT (-38.51083 -12.97111)
1,2,"Bahia, port unspecified",Tibério,1816-20,1811-20,1801-25,1800,Portugal / Brazil,1816,1817,...,NaN,0.0,0.0,0.0,0.0,0.0,0.0,-38.51083,-12.97111,POINT (-38.51083 -12.97111)
2,3,"Bahia, port unspecified",Paquete Real,1816-20,1811-20,1801-25,1800,Portugal / Brazil,1816,1817,...,NaN,0.0,0.0,0.0,0.0,0.0,0.0,-38.51083,-12.97111,POINT (-38.51083 -12.97111)


In [22]:
# 1. Read the dataset
dataset = "SLA1PORT"
ports_sel = gp.read_file(f"data\\{dataset}.gpkg")

# Define the periods
periods = [(1500, 1600), (1600, 1700), (1700, 1800), (1800, 1900), (1800, 1806), (1807, 1832), (1833, 1900)]

for start, end in periods:
    # Filter the dataset for the given period
    period_sel = ports_sel[(ports_sel['YEARDEP'] >= start) & (ports_sel['YEARDEP'] < end)]
    
    # 2. One-hot encode the 'NATINIMP' column
    country_counts = pd.get_dummies(period_sel[[dataset, 'NATINIMP', 'YEARDEP', dataset+'_lng', dataset+'_lat']], columns=['NATINIMP'])

    # 3. Group by 'ADPSALE2' and aggregate the columns
    country_sums = country_counts.groupby(dataset).agg({
        **{col: 'sum' for col in country_counts.columns if col.startswith('NATINIMP_')},  # Sum for one-hot encoded columns
        dataset+'_lng': 'first',  # Keep the first entry of the longitude column
        dataset+'_lat': 'first',  # Keep the first entry of the latitude column
        'YEARDEP': 'first'  # Keep the first entry of the YEARDEP column
    }).reset_index()

    # 4. Add a column to count all countries together
    country_columns = [col for col in country_sums.columns if col.startswith('NATINIMP_')]
    country_sums['countries_SUM'] = country_sums[country_columns].sum(axis=1)

    # 5. Rename columns by removing the "NATINIMP_" prefix
    country_sums = country_sums.rename(
        columns=lambda col: col.replace("NATINIMP_", "") if col.startswith("NATINIMP_") else col
    )

    # 6. Convert DataFrame to GeoDataFrame
    gdf = gp.GeoDataFrame(
        country_sums,
        geometry=gp.points_from_xy(country_sums[dataset+'_lng'], country_sums[dataset+'_lat']),
        crs="EPSG:4326"
    )

    # 7. Export the GeoDataFrame to a GeoPackage for the period
    output_path = f"data\\{dataset}_ports_{start}_{end}.gpkg"
    gdf.to_file(output_path, driver="GPKG")
    print(f"Processed and saved: {output_path}")

Processed and saved: data\SLA1PORT_ports_1500_1600.gpkg
Processed and saved: data\SLA1PORT_ports_1600_1700.gpkg
Processed and saved: data\SLA1PORT_ports_1700_1800.gpkg
Processed and saved: data\SLA1PORT_ports_1800_1900.gpkg
Processed and saved: data\SLA1PORT_ports_1800_1806.gpkg
Processed and saved: data\SLA1PORT_ports_1807_1832.gpkg
Processed and saved: data\SLA1PORT_ports_1833_1900.gpkg


In [5]:
# 1. Read the dataset
ports_sel = gp.read_file(f"data\\SLA1PORT.gpkg")

In [7]:
ports_sel.info()

<class 'geopandas.geodataframe.GeoDataFrame'>
RangeIndex: 25198 entries, 0 to 25197
Data columns (total 41 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   VOYAGEID            25198 non-null  int32         
 1   SLA1PORT            25198 non-null  object        
 2   SHIPNAME            25198 non-null  object        
 3   YEAR5               25198 non-null  object        
 4   YEAR10              25198 non-null  object        
 5   YEAR25              25198 non-null  object        
 6   YEAR100             25198 non-null  int32         
 7   NATINIMP            25198 non-null  object        
 8   YEARDEP             25198 non-null  int32         
 9   YEARAM              25198 non-null  int32         
 10  CAPTAINA            25198 non-null  object        
 11  CAPTAINB            25198 non-null  object        
 12  CAPTAINC            25198 non-null  object        
 13  JAMCASPR            941 non-null    fl